# VinFast DMS — Google Colab Webcam

### Cách chạy (QUAN TRỌNG)
1. Chạy **Cell 1** để cài thư viện (chạy một lần cho mỗi runtime Colab).
2. Chạy **Cell 2a**, **Cell 2b** để tải model và nạp thuật toán.
3. Chạy **Cell 3**. Nhấn nút **START webcam + AI**, cấp quyền camera. **Bạn KHÔNG CẦN chạy Cell 4 để bắt đầu nhận diện.**
4. Nhìn thẳng khi hiệu chuẩn, rồi thử nhìn thẳng / nói chuyện / nói + quay đầu. Các nút `1/2/3` dùng gắn nhãn kiểm chứng, **không điều khiển dự đoán**.
5. Nhấn **STOP webcam** (hoặc chờ hết `MAX_SECONDS`); chạy **Cell 4** để thống kê và tải CSV.

Model là MediaPipe Face Landmarker pretrained, kết hợp tín hiệu miệng + hình học quay đầu + ngưỡng thời gian. Cảnh báo là **prototype cho hackathon**, không khẳng định nhận biết tiếng nói thật (không dùng microphone) hoặc mức độ mất tập trung trong xe thật.

Nếu FPS còn 0 sau khi START, đọc dòng báo lỗi bên dưới khung video; kiểm tra đã chạy Cell 1–3 và không dùng **Run all** khi webcam đang chạy. Dùng Chrome/Edge và môi trường Colab.

In [1]:
%pip -q install "mediapipe==0.10.35" "opencv-python-headless<5" "numpy<3" pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 12.9 MB/s eta 0:00:00


In [2]:
from pathlib import Path
from urllib.request import urlretrieve
from IPython.display import display, Javascript
from google.colab.output import eval_js
import base64, json, time, math, io
import cv2, mediapipe as mp, numpy as np, pandas as pd

MODEL_FILE = Path('/content/face_landmarker.task')
MODEL_URL = 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task'
if not MODEL_FILE.exists() or MODEL_FILE.stat().st_size < 100000:
    print('Đang tải model Face Landmarker từ Google...')
    urlretrieve(MODEL_URL, MODEL_FILE)
print(f'MediaPipe {mp.__version__}, model = {MODEL_FILE}, size = {MODEL_FILE.stat().st_size/1e6:.2f} MB')


Đang tải model Face Landmarker từ Google...
MediaPipe 0.10.35, model = /content/face_landmarker.task, size = 3.76 MB


In [3]:
from collections import deque
import numpy as np


class DriverMonitor:
    def __init__(self, mouth_variation_threshold=0.085, yaw_threshold=0.22,
                 pitch_threshold=0.16, combo_seconds=0.8, away_seconds=2.0):
        self.mouth_variation_threshold = float(mouth_variation_threshold)
        self.yaw_threshold = float(yaw_threshold)
        self.pitch_threshold = float(pitch_threshold)
        self.combo_seconds = float(combo_seconds)
        self.away_seconds = float(away_seconds)
        self.episodes = 0
        self.last_alert = False
        self.reset_calibration()

    def reset_calibration(self):
        self.calibration = []
        self.reference = None
        self.mouth_history = deque(maxlen=120)
        self.away_since = None
        self.combo_since = None
        self.last_alert = False

    def analyze(self, landmarks, blendshapes, timestamp_ms):
        t = timestamp_ms / 1000.0
        base = dict(state='NO_FACE', speaking=False, head_away=False,
                    alert=False, calibrated=self.reference is not None,
                    yaw_delta=0.0, pitch_delta=0.0, mouth_signal=0.0,
                    mouth_variation=0.0, jaw_open=0.0, mar=0.0,
                    bbox=None, points=[], episodes=self.episodes)
        if not landmarks:
            self.mouth_history.clear()
            self.away_since = self.combo_since = None
            self.last_alert = False
            return base

        def pt(i):
            return np.array([landmarks[i].x, landmarks[i].y], dtype=float)

        eye_l, eye_r = pt(33), pt(263)
        eye_mid = (eye_l + eye_r) / 2.0
        nose, chin = pt(1), pt(152)
        eye_width = max(float(np.linalg.norm(eye_r-eye_l)), 1e-4)
        yaw_proxy = float(2.0 * (nose[0]-eye_mid[0]) / eye_width)
        pitch_proxy = float((nose[1]-eye_mid[1]) / max(abs(chin[1]-eye_mid[1]), 1e-4))
        mouth_width = max(float(np.linalg.norm(pt(61)-pt(291))), 1e-4)
        mar = float(np.linalg.norm(pt(13)-pt(14)) / mouth_width)
        shapes = {c.category_name: float(c.score) for c in (blendshapes or [])}
        jaw_open = float(shapes.get('jawOpen', 0.0))
        mouth_signal = float(0.5*min(1.0, 3.0*mar) + 0.5*jaw_open)

        xs = [max(0., min(1., float(p.x))) for p in landmarks]
        ys = [max(0., min(1., float(p.y))) for p in landmarks]
        base.update(mouth_signal=mouth_signal, jaw_open=jaw_open, mar=mar,
                    bbox=[min(xs), min(ys), max(xs), max(ys)],
                    points=[[float(landmarks[j].x), float(landmarks[j].y)]
                            for j in (33, 263, 1, 13, 14)])

        if self.reference is None:
            self.calibration.append((yaw_proxy, pitch_proxy))
            if len(self.calibration) >= 12:
                self.reference = np.median(np.array(self.calibration), axis=0)
                self.mouth_history.clear()
            base.update(state='CALIBRATING', calibrated=False)
            return base

        yaw_delta = abs(yaw_proxy - self.reference[0])
        pitch_delta = abs(pitch_proxy - self.reference[1])
        head_away = bool(yaw_delta > self.yaw_threshold or
                         pitch_delta > self.pitch_threshold)

        self.mouth_history.append((t, mouth_signal))
        while self.mouth_history and t-self.mouth_history[0][0] > 2.4:
            self.mouth_history.popleft()
        values = np.array([val for _, val in self.mouth_history])
        span = t-self.mouth_history[0][0] if self.mouth_history else 0.0
        variation = float(np.percentile(values, 90)-np.percentile(values, 10)) if len(values) >= 2 else 0.0
        motions = int(np.sum(np.abs(np.diff(values)) > 0.035)) if len(values) >= 2 else 0
        speaking = bool(len(values) >= 5 and span >= 0.8 and
                        variation >= self.mouth_variation_threshold and motions >= 2)

        if head_away:
            if self.away_since is None:
                self.away_since = t
        else:
            self.away_since = None
        if head_away and speaking:
            if self.combo_since is None:
                self.combo_since = t
        else:
            self.combo_since = None

        combo_alert = (self.combo_since is not None and t-self.combo_since >= self.combo_seconds)
        away_alert = (self.away_since is not None and t-self.away_since >= self.away_seconds)
        alert = bool(combo_alert or away_alert)
        if alert and not self.last_alert:
            self.episodes += 1
        self.last_alert = alert

        if combo_alert:
            state = 'ALERT_TALK_AWAY'
        elif away_alert:
            state = 'ALERT_AWAY'
        elif speaking and head_away:
            state = 'TALK_AWAY_PENDING'
        elif speaking:
            state = 'POSSIBLE_TALKING'
        elif head_away:
            state = 'LOOKING_AWAY'
        else:
            state = 'FOCUSED'
        base.update(state=state, speaking=speaking, head_away=head_away,
                    alert=alert, calibrated=True,
                    yaw_delta=round(float(yaw_delta), 4),
                    pitch_delta=round(float(pitch_delta), 4),
                    mouth_variation=round(variation, 4),
                    episodes=self.episodes)
        return base


## Cell 3 — Giao diện + AI tự chạy

Nhấn Run của cell dưới đây để **đăng ký Python callback** và hiển thị webcam. Khi bạn bấm **START webcam + AI**, JavaScript tự gửi khung hình tới Python và nhận dự đoán để vẽ lên canvas. **Không chạy thêm cell xử lý đồng thời.**

Bạn có thể thay `MAX_SECONDS = 120` thành `180` hoặc `300` ở cell này trước khi chạy.

In [6]:
from google.colab import output
from IPython.display import display, Javascript, JSON

MAX_SECONDS = 120

options = mp.tasks.vision.FaceLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL_FILE)),
    running_mode=mp.tasks.vision.RunningMode.VIDEO,
    num_faces=1,
    output_face_blendshapes=True,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)
try:
    detector.close()
except (NameError, AttributeError):
    pass

detector = mp.tasks.vision.FaceLandmarker.create_from_options(options)
monitor = DriverMonitor()
records = []
time_history = []
last_ts = -1
first_ts = None


def reset_demo():
    global monitor, records, time_history, first_ts
    monitor = DriverMonitor()
    records = []
    time_history = []
    first_ts = None
    return JSON({'ok': True, 'status': 'Ready'})


def process_camera_frame(packet):
    global last_ts, first_ts, time_history
    started = time.perf_counter()
    url = packet.get('data_url', '')
    if not url or ',' not in url:
        return JSON({'state': 'WAITING', 'fps': 0, 'inference_ms': 0})
    encoded = url.split(',', 1)[1]
    jpeg = np.frombuffer(base64.b64decode(encoded), dtype=np.uint8)
    image_bgr = cv2.imdecode(jpeg, cv2.IMREAD_COLOR)
    if image_bgr is None:
        return JSON({'state': 'WAITING', 'fps': 0, 'inference_ms': 0})
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
    timestamp_ms = max(last_ts + 1, int(packet['timestamp_ms']))
    last_ts = timestamp_ms
    if first_ts is None:
        first_ts = timestamp_ms
    result = detector.detect_for_video(image, timestamp_ms)
    landmarks = result.face_landmarks[0] if result.face_landmarks else None
    blends = result.face_blendshapes[0] if result.face_blendshapes else None
    if packet.get('reset'):
        monitor.reset_calibration()
    detection = monitor.analyze(landmarks, blends, timestamp_ms)
    inference_ms = (time.perf_counter() - started) * 1000

    time_history.append(timestamp_ms)
    time_history = time_history[-12:]
    fps = (1000 * (len(time_history) - 1) /
           (time_history[-1] - time_history[0])
           if len(time_history) > 1 and time_history[-1] > time_history[0]
           else 0.0)
    prediction = {**detection, 'fps': round(fps, 1),
                  'inference_ms': round(inference_ms, 1)}
    records.append({
        'time_s': round((timestamp_ms - first_ts) / 1000, 3),
        'timestamp_ms': timestamp_ms,
        'ground_truth': packet.get('gt', 'unlabeled'),
        'state': detection['state'],
        'calibrated': detection['calibrated'],
        'speaking': detection['speaking'],
        'head_away': detection['head_away'],
        'alert': detection['alert'],
        'alert_episodes': detection['episodes'],
        'mouth_signal': detection['mouth_signal'],
        'mouth_variation': detection['mouth_variation'],
        'yaw_delta': detection['yaw_delta'],
        'pitch_delta': detection['pitch_delta'],
        'inference_ms': round(inference_ms, 2),
        'sampling_fps': round(fps, 2),
    })
    return JSON(prediction)


output.register_callback('vinfast_dms.reset', reset_demo)
output.register_callback('vinfast_dms.frame', process_camera_frame)
print('Python AI đã sẵn sàng. Bấm START webcam + AI bên dưới; không cần chạy thêm cell realtime.')
frontend_js = r'''// v2: Browser-owned video loop. Frames are sent directly to registered
// Python callbacks using Colab's supported invokeFunction API.
// No eval_js() lookup across output iframe boundaries.
(() => {
  const prior = document.getElementById('vin-dms-demo-v2');
  if (prior) prior.remove();
  const root = document.createElement('div');
  root.id = 'vin-dms-demo-v2';
  root.innerHTML = `
    <style>
      #vin-dms-demo-v2 {font:14px Arial,sans-serif;max-width:760px;background:#101d2c;color:#f2f6fc;border-radius:12px;padding:16px}
      #vin-dms-demo-v2 button {margin:4px;padding:9px 12px;border:0;border-radius:7px;cursor:pointer;background:#dce8fc;color:#18283b;font-weight:bold}
      #vin-dms-demo-v2 button:hover {background:#b6d0ff}
      #vin-dms-demo-v2 button:disabled {opacity:.55;cursor:not-allowed}
      #vin-dms-demo-v2 canvas {width:100%;max-width:640px;border-radius:8px;background:#23394f;display:block;margin:10px auto}
      #vin-dms-demo-v2 .muted {font-size:12px;color:#aec4d8;margin:8px 0}
      #vin-dms-demo-v2 .line {margin:7px 0;display:flex;align-items:center;flex-wrap:wrap}
      #vin-dms-demo-v2 .tag {margin-right:12px;font-size:13px}
    </style>
    <h3 style="margin:0 0 4px">VinFast Driver Monitoring · Colab webcam (v2)</h3>
    <p class="muted">Chạy Cell 1, Cell 2a, Cell 2b, Cell 3. Sau đó chỉ cần START webcam: AI tự xử lý. Nhìn thẳng trong 12 khung hình đầu để hiệu chuẩn.</p>
    <div class="line">
      <button id="vin-start">▶ START webcam + AI</button>
      <button id="vin-stop">■ STOP webcam</button>
      <button id="vin-calib">⟳ Hiệu chuẩn lại</button>
      <button id="vin-rec">● Quay video dự phòng</button>
    </div>
    <video id="vin-video" autoplay playsinline muted style="display:none"></video>
    <canvas id="vin-canvas" width="640" height="480"></canvas>
    <div class="line"><span class="tag">Nhãn thủ công để đánh giá:</span>
      <button class="vin-label" data-value="focus">1 · Tập trung</button>
      <button class="vin-label" data-value="talk">2 · Chỉ nói chuyện</button>
      <button class="vin-label" data-value="distracted">3 · Nói + quay đầu</button>
    </div>
    <p id="vin-message" class="muted">Chưa bật webcam. Không cần chạy một Cell 4 để xử lý realtime.</p>
    <p class="muted">Sau khi bấm STOP, chạy Cell 4 để xem số liệu và tải CSV. Không bấm Run all khi camera đang chạy.</p>`;
  document.body.appendChild(root);
  try { google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true); } catch (_) {}

  const video = root.querySelector('#vin-video');
  const canvas = root.querySelector('#vin-canvas');
  const ctx = canvas.getContext('2d');
  const grab = document.createElement('canvas');
  grab.width = 640; grab.height = 480;
  const grabCtx = grab.getContext('2d', {willReadFrequently:true});
  const msg = root.querySelector('#vin-message');
  const recordBtn = root.querySelector('#vin-rec');
  const startBtn = root.querySelector('#vin-start');
  const state = {running:false, gt:'unlabeled', prediction:{state:'WAITING'},
    reset:true, stream:null, recorder:null, chunks:[], startedAt:0,
    busy:false, sessionId:0};
  const labels = {
    WAITING:'CHỜ MÔ HÌNH', NO_FACE:'KHÔNG THẤY KHUÔN MẶT', CALIBRATING:'ĐANG HIỆU CHUẨN',
    FOCUSED:'ĐANG TẬP TRUNG', POSSIBLE_TALKING:'NGHI ĐANG NÓI CHUYỆN',
    LOOKING_AWAY:'ĐANG QUAY ĐẦU', TALK_AWAY_PENDING:'NÓI + QUAY ĐẦU',
    ALERT_AWAY:'CẢNH BÁO: QUAY ĐẦU LÂU', ALERT_TALK_AWAY:'CẢNH BÁO: NÓI + QUAY ĐẦU'
  };

  async function callPython(name, args=[]) {
    if (!window.google?.colab?.kernel?.invokeFunction) {
      throw new Error('Không tìm thấy cầu nối Colab. Hãy mở notebook trong Google Colab rồi chạy lại Cell 3.');
    }
    const response = await google.colab.kernel.invokeFunction(name, args, {});
    const data = response?.data?.['application/json'];
    if (!data) throw new Error('Python không trả JSON; xem lỗi callback trong Colab.');
    return data;
  }

  async function startCamera() {
    if (state.running || state.busy) return;
    state.busy = true;
    startBtn.disabled = true;
    try {
      if (!navigator.mediaDevices?.getUserMedia) throw Error('Không hỗ trợ getUserMedia.');
      msg.textContent = 'Đang khởi tạo AI trong Python...';
      await callPython('vinfast_dms.reset', []);
      state.stream = await navigator.mediaDevices.getUserMedia({
        video:{width:{ideal:640},height:{ideal:480},facingMode:'user'},audio:false
      });
      video.srcObject = state.stream;
      await video.play();
      state.running = true;
      state.reset = true;
      state.prediction = {state:'CALIBRATING'};
      state.startedAt = performance.now();
      state.sessionId += 1;
      msg.textContent = 'Webcam + AI đã bật. Nhìn thẳng vào camera để hiệu chuẩn.';
      pump(state.sessionId);
    } catch (e) {
      if (state.stream) state.stream.getTracks().forEach(t=>t.stop());
      state.stream = null;
      state.running = false;
      msg.textContent = 'Lỗi START webcam/AI: ' + String(e);
    } finally {
      state.busy = false;
      startBtn.disabled = false;
    }
  }

  function stopCamera(reason) {
    state.running = false;
    state.sessionId += 1;
    if (state.recorder?.state === 'recording') state.recorder.stop();
    if (state.stream) state.stream.getTracks().forEach(t=>t.stop());
    state.stream = null;
    video.srcObject = null;
    msg.textContent = reason || 'Đã dừng. Chạy Cell 4 để xem số liệu và tải CSV.';
  }

  async function pump(mySession) {
    // The capture, Python inference and response run serially: no backpressure queue.
    while (state.running && mySession === state.sessionId) {
      if (performance.now() - state.startedAt > MAX_DEMO_SECONDS * 1000) {
        stopCamera('Đã hết thời gian demo. Chạy Cell 4 để xem log.');
        break;
      }
      if (video.readyState < 2) {
        await new Promise(r=>setTimeout(r,100));
        continue;
      }
      try {
        grabCtx.drawImage(video,0,0,640,480);
        const packet = {
          data_url:grab.toDataURL('image/jpeg',0.65),
          timestamp_ms:performance.now(), gt:state.gt, reset:state.reset
        };
        state.reset = false;
        const result = await callPython('vinfast_dms.frame', [packet]);
        if (state.running && mySession === state.sessionId) state.prediction = result;
      } catch (e) {
        stopCamera('Lỗi xử lý AI: '+String(e)+'; kiểm tra output của Cell 3.');
        break;
      }
      await new Promise(r=>setTimeout(r,220));
    }
  }

  function recordVideo() {
    if (state.recorder?.state === 'recording') {
      state.recorder.stop();recordBtn.textContent='● Quay video dự phòng';return;
    }
    if (!state.running) {msg.textContent='Hãy START webcam trước.';return;}
    try {
      if (!window.MediaRecorder || !canvas.captureStream) throw Error('Trình duyệt không hỗ trợ quay canvas.');
      const mime = ['video/webm;codecs=vp8','video/webm'].find(t=>MediaRecorder.isTypeSupported(t));
      const recorder = new MediaRecorder(canvas.captureStream(12),mime?{mimeType:mime}:{});
      state.chunks = [];
      recorder.ondataavailable = e=>{if(e.data?.size)state.chunks.push(e.data);};
      recorder.onstop=()=>{
        const url=URL.createObjectURL(new Blob(state.chunks,{type:'video/webm'}));
        const a=document.createElement('a'); a.href=url;a.download='vinfast_dms_demo_backup.webm';
        document.body.appendChild(a);a.click();a.remove();
        setTimeout(()=>URL.revokeObjectURL(url),10000);
        recordBtn.textContent='● Quay video dự phòng';
        msg.textContent='Đã tạo video .webm. Kiểm tra thư mục Downloads.';
      };
      state.recorder = recorder;
      recorder.start(1000);
      recordBtn.textContent='■ Dừng quay và lưu';
      msg.textContent='Đang quay video dự phòng có cảnh báo trên canvas.';
    } catch(e) {msg.textContent='Lỗi quay video: '+String(e);}
  }

  root.querySelector('#vin-start').onclick=startCamera;
  root.querySelector('#vin-stop').onclick=()=>stopCamera();
  root.querySelector('#vin-calib').onclick=()=>{
    state.reset=true;state.prediction={state:'CALIBRATING'};
    msg.textContent='Đang yêu cầu hiệu chuẩn lại. Nhìn thẳng vào camera.';
  };
  root.querySelector('#vin-rec').onclick=recordVideo;
  root.querySelectorAll('.vin-label').forEach(btn=>btn.onclick=()=>{
    state.gt=btn.dataset.value;
    root.querySelectorAll('.vin-label').forEach(b=>b.style.background=b===btn?'#a8f0b7':'#dce8fc');
    msg.textContent='Nhãn thủ công: '+state.gt+' (nhãn không điều khiển dự đoán).';
  });

  function draw() {
    if (state.running && video.readyState >= 2) {
      ctx.save();ctx.translate(640,0);ctx.scale(-1,1);ctx.drawImage(video,0,0,640,480);ctx.restore();
    } else {
      ctx.fillStyle='#26394d';ctx.fillRect(0,0,640,480);
      ctx.fillStyle='#b4c7d9';ctx.font='24px Arial';ctx.fillText('Webcam preview',205,250);
    }
    const p=state.prediction||{};
    const danger=!!p.alert;
    const amber=['POSSIBLE_TALKING','LOOKING_AWAY','TALK_AWAY_PENDING'].includes(p.state);
    const accent=danger?'#ff616a':amber?'#ffcc60':'#66e2af';
    if (p.bbox) {
      const [x0,y0,x1,y1]=p.bbox;
      ctx.strokeStyle=accent;ctx.lineWidth=3;
      ctx.strokeRect((1-x1)*640,y0*480,(x1-x0)*640,(y1-y0)*480);
      if (p.points) for (const [x,y] of p.points) {
        ctx.beginPath();ctx.arc((1-x)*640,y*480,3,0,Math.PI*2);
        ctx.fillStyle=accent;ctx.fill();
      }
    }
    ctx.fillStyle='rgba(9,23,37,.85)';ctx.fillRect(0,0,640,84);
    ctx.fillStyle=accent;ctx.font='bold 22px Arial';
    ctx.fillText(labels[p.state]||p.state||'CHỜ MÔ HÌNH',14,30);
    ctx.fillStyle='#f7fafc';ctx.font='14px Arial';
    ctx.fillText(`FPS ${Number(p.fps||0).toFixed(1)}  |  Inference ${Number(p.inference_ms||0).toFixed(0)} ms  |  Cảnh báo ${p.episodes||0}`,14,54);
    ctx.fillText(`Môi Δ ${Number(p.mouth_variation||0).toFixed(3)}  |  Hướng Δ ${Number(p.yaw_delta||0).toFixed(2)}  |  Nhãn ${state.gt}`,14,75);
    requestAnimationFrame(draw);
  }
  requestAnimationFrame(draw);
})();
'''
display(Javascript('const MAX_DEMO_SECONDS = ' + str(MAX_SECONDS) + ';\n' + frontend_js))


Python AI đã sẵn sàng. Bấm START webcam + AI bên dưới; không cần chạy thêm cell realtime.


<IPython.core.display.Javascript object>

## Cell 4 — Thống kê và tải CSV (chỉ chạy SAU KHI bấm STOP)

Mỗi frame đã xử lý có một dòng trong `records`. Chạy cell dưới đây để tính độ trễ, FPS và precision/recall trên dữ liệu **tự gắn nhãn** trong buổi demo.

In [8]:
from google.colab import files
log_path = '/content/vinfast_dms_log.csv'
df = pd.DataFrame(records)
if df.empty:
    print('Chưa nhận được frame nào. Kiểm tra bạn đã bấm START webcam + AI ở Cell 3.')
else:
    df.to_csv(log_path, index=False, encoding='utf-8-sig')
    print('Số frame được xử lý:', len(df))
    print('Inference trung bình: %.1f ms / P95: %.1f ms' %
          (df.inference_ms.mean(), df.inference_ms.quantile(0.95)))
    measured = df.sampling_fps[df.sampling_fps > 0]
    if len(measured):
        print('FPS trung bình:', round(float(measured.mean()), 2))
    print('Số đợt cảnh báo:', int(df.alert_episodes.max()))
    print('\nPhân bố trạng thái:')
    display(df.state.value_counts().rename('Frames').to_frame())

    valid = df[(df.ground_truth.isin(['focus','talk','distracted'])) &
               df.calibrated & df.state.ne('NO_FACE')]
    if not valid.empty:
        pos = valid.ground_truth.eq('distracted')
        pred = valid.alert.astype(bool)
        tp = int((pos & pred).sum())
        fp = int((~pos & pred).sum())
        fn = int((pos & ~pred).sum())
        tn = int((~pos & ~pred).sum())
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        print(f'\nTP={tp}, FP={fp}, FN={fn}, TN={tn}')
        print(f'Precision={precision:.1%} | Recall={recall:.1%} (chỉ demo theo frame)')
    else:
        print('Chưa có nhãn thủ công đủ để tính precision/recall.')
    print('\nCa lỗi cần giải thích: nhai hoặc há miệng kết hợp quay đầu có thể gây báo nhầm.')
    print('CSV:', log_path)
    files.download(log_path)


Số frame được xử lý: 116
Inference trung bình: 20.8 ms / P95: 24.4 ms
FPS trung bình: 2.79
Số đợt cảnh báo: 1

Phân bố trạng thái:


,Frames
state,
FOCUSED,37
POSSIBLE_TALKING,30
LOOKING_AWAY,20
CALIBRATING,12
NO_FACE,10
TALK_AWAY_PENDING,6
ALERT_AWAY,1


Chưa có nhãn thủ công đủ để tính precision/recall.

Ca lỗi cần giải thích: nhai hoặc há miệng kết hợp quay đầu có thể gây báo nhầm.
CSV: /content/vinfast_dms_log.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>